In [1]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from qdrant_client import QdrantClient, models
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct
)

## 1. Build index
- Load company documents
- Generate BGE embeddings
- Create a Qdrant collection
- Store embeddings

In [2]:

# =====================================================
# Sample Documents
# =====================================================

documents = [

"""
Apple Inc.

Apple develops the iPhone, MacBook, iPad and Apple Watch.

Apple develops iOS and macOS.

Apple also offers iCloud and Apple Music.
""",

"""
Microsoft Corporation

Microsoft develops Windows and Microsoft Office.

Microsoft Azure is a cloud computing platform.

Microsoft acquired GitHub in 2018.
""",

"""
Google

Google develops Search, Gmail, Chrome and Android.

Google Cloud provides cloud computing services.

Google invests heavily in Artificial Intelligence.
""",

"""
Amazon

Amazon operates one of the world's largest online marketplaces.

Amazon Web Services (AWS) provides cloud computing.

Amazon manufactures Kindle and Echo devices.
""",

"""
Tesla

Tesla manufactures electric vehicles.

Popular vehicles include Model S, Model 3 and Cybertruck.

Tesla also develops battery technology.
"""

]

In [3]:
# =====================================================
# Load BGE Embedding Model
# =====================================================

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

print("Model Loaded")


Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model Loaded


In [4]:
# =====================================================
# Create Embeddings
# =====================================================

print("\nGenerating embeddings...")

embeddings = embedding_model.encode(
    documents,
    normalize_embeddings=True
)

print(embeddings)

print("Embedding Shape:", embeddings.shape)


Generating embeddings...
[[-0.01020336  0.04246262  0.00888172 ...  0.00678466 -0.01328184
  -0.03596935]
 [ 0.01408674  0.04792781  0.01827758 ... -0.01060778  0.02027568
   0.03628571]
 [-0.00376575  0.02487622  0.02562853 ... -0.0311626   0.03475658
   0.01361183]
 [-0.01806602  0.00509413  0.01328153 ...  0.01240794  0.07521465
  -0.01681308]
 [ 0.04953429  0.00484357 -0.00118785 ... -0.01283366  0.07241265
  -0.03508963]]
Embedding Shape: (5, 768)


In [5]:
# =====================================================
# Create Qdrant Database
# =====================================================

client = QdrantClient(path="./qdrant_data")

COLLECTION_NAME = "companies"

# Delete collection if already exists

try:
    client.delete_collection(COLLECTION_NAME)
except:
    pass

client.create_collection(

    collection_name=COLLECTION_NAME,

    vectors_config=VectorParams(

        size=embeddings.shape[1],

        distance=Distance.COSINE

    )

)

print("\nCollection Created")


Collection Created


In [6]:
# =====================================================
# Store Documents
# =====================================================

points = []

for i in range(len(documents)):

    point = PointStruct(

        id=i,

        vector=embeddings[i].tolist(),

        payload={

            "text": documents[i]

        }

    )

    points.append(point)

client.upsert(

    collection_name=COLLECTION_NAME,

    points=points

)

print("\nStored", len(points), "documents.")


points, _ = client.scroll(
    collection_name="companies",
    limit=100,
    with_vectors=True,
    with_payload=True
)

for point in points:
    print("ID:", point.id)
    print("Payload:", point.payload)
    print("Vector length:", len(point.vector))
    print("Vector", point.vector)
    print("-" * 60)



Stored 5 documents.
ID: 0
Payload: {'text': '\nApple Inc.\n\nApple develops the iPhone, MacBook, iPad and Apple Watch.\n\nApple develops iOS and macOS.\n\nApple also offers iCloud and Apple Music.\n'}
Vector length: 768
Vector [-0.010203361136665914, 0.04246262305067094, 0.008881716662933676, 0.05740983200918706, 0.08145580141736077, 0.02363319421974102, 0.02287028459694476, 0.06258670010885374, -0.03314563510334057, -0.04603770951305774, -0.004189594029317496, -0.0072835411065303194, -0.041137108931783226, 0.02972121613889893, -0.0027329734454624864, 0.024571414134902124, 0.05452483319751548, 0.025001474669685126, 0.004709143484365018, 0.02096455653537121, -0.019048524321210382, 0.030303002154033004, -0.00040468298508657233, 0.010927440197461625, 0.023038900542889013, -0.009426314969028514, 0.019341322814695132, -0.0015790858890462772, -0.034255119645750905, -0.02545162383166513, 0.02704635535731399, 0.024174141741477386, -0.02415403076253941, -0.013237179702362439, -0.04779294702796

In [7]:
# =====================================================
# Verify
# =====================================================

count = client.count(
    collection_name=COLLECTION_NAME,
    exact=True
)

print("\nDocuments in Qdrant:", count.count)

print("\nIndex creation completed.")


Documents in Qdrant: 5

Index creation completed.


## 2. Retrieve embeddings

- Connects to the existing Qdrant database
- Loads the same BGE embedding model
- Converts the user's query into an embedding
- Retrieves the Top-K most similar documents

In [ ]:
query = input("\nEnter your question: ")

# For BGE models, adding a query prefix improves retrieval
query_embedding = embedding_model.encode(
    "Represent this sentence for searching relevant passages: " + query,
    normalize_embeddings=True
)

In [9]:
# =====================================================
# Semantic Search
# =====================================================

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_embedding.tolist(),
    limit=3
)


In [10]:
# =====================================================
# Display Results
# =====================================================

print("\n==============================")
print("Top Matching Documents")
print("==============================")

retrieved_docs = []

for rank, point in enumerate(results.points, start=1):

    retrieved_docs.append(point.payload["text"])

    print(f"\nRank : {rank}")
    print(f"Score: {point.score:.4f}")
    print("-" * 40)
    print(point.payload["text"])



Top Matching Documents

Rank : 1
Score: 0.6451
----------------------------------------

Tesla

Tesla manufactures electric vehicles.

Popular vehicles include Model S, Model 3 and Cybertruck.

Tesla also develops battery technology.


Rank : 2
Score: 0.4255
----------------------------------------

Apple Inc.

Apple develops the iPhone, MacBook, iPad and Apple Watch.

Apple develops iOS and macOS.

Apple also offers iCloud and Apple Music.


Rank : 3
Score: 0.4084
----------------------------------------

Google

Google develops Search, Gmail, Chrome and Android.

Google Cloud provides cloud computing services.

Google invests heavily in Artificial Intelligence.



## 3. Reranking

- Load the BGE reranker
- Rerank the retrieved documents
- Keep the best documents

In [11]:
# =====================================================
# Load BGE Reranker
# =====================================================

print("Loading BGE Reranker...")

reranker = CrossEncoder(
    "BAAI/bge-reranker-base"
)

print("Model Loaded")

# =====================================================
# Create Query-Document Pairs
# =====================================================

pairs = []

for doc in retrieved_docs:
    pairs.append([query, doc])

# =====================================================
# Predict Relevance Scores
# =====================================================

scores = reranker.predict(pairs)

# =====================================================
# Combine Documents + Scores
# =====================================================

results = list(zip(retrieved_docs, scores))

# Sort by highest score

results.sort(
    key=lambda x: x[1],
    reverse=True
)

# =====================================================
# Display Results
# =====================================================

print("\n==============================")
print("Reranked Documents")
print("==============================")

for rank, (doc, score) in enumerate(results, start=1):

    print(f"\nRank : {rank}")
    print(f"Reranker Score : {score:.4f}")
    print("-" * 40)
    print(doc)

# =====================================================
# Keep Top 2 Documents
# =====================================================

top_docs = [doc for doc, score in results[:2]]

print("\n==============================")
print("Documents Sent To LLM")
print("==============================")

for doc in top_docs:
    print(doc)
    print("-" * 60)

Loading BGE Reranker...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model Loaded

Reranked Documents

Rank : 1
Reranker Score : 0.3517
----------------------------------------

Tesla

Tesla manufactures electric vehicles.

Popular vehicles include Model S, Model 3 and Cybertruck.

Tesla also develops battery technology.


Rank : 2
Reranker Score : 0.0000
----------------------------------------

Google

Google develops Search, Gmail, Chrome and Android.

Google Cloud provides cloud computing services.

Google invests heavily in Artificial Intelligence.


Rank : 3
Reranker Score : 0.0000
----------------------------------------

Apple Inc.

Apple develops the iPhone, MacBook, iPad and Apple Watch.

Apple develops iOS and macOS.

Apple also offers iCloud and Apple Music.


Documents Sent To LLM

Tesla

Tesla manufactures electric vehicles.

Popular vehicles include Model S, Model 3 and Cybertruck.

Tesla also develops battery technology.

------------------------------------------------------------

Google

Google develops Search, Gmail, Chrome and Andro

## 4. Prompting

- Build the final prompt
- Send the prompt to AutoModelForSeq2SeqLM
- Print the answer

In [ ]:
# =====================================================
# Build Context
# =====================================================


tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

context = "\n\n".join(top_docs)


# =====================================================
# Build Prompt
# =====================================================

prompt = f"""
You are a helpful assistant.

Answer ONLY using the context below.

If the answer is not present, reply:

"I don't know based on the supplied context."

----------------------------

Context:

{context}

----------------------------

Question:

{query}

Answer:
"""

# =====================================================
# Load LLM
# =====================================================

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True
)

outputs = model.generate(
    **inputs,
    max_new_tokens=50
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

# =====================================================
# Final Output
# =====================================================

print("\n")

print("Question:", query)

print("Answer:", answer)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
